**Eye-Tracking & Neurocognitive Intelligence | Eye-Movement Modelling & Advanced AI | Presented by: Reza Saadatyar (2026)**

**02 | Statistical Baselines and Classical Machine Learning**

![Study 2 feature-based task classification and four separate model comparisons](../../images/02_statistical_and_classical_ml.svg)

**Objective:** Compare a no-signal class-prior predictor with Logistic Regression, Random Forest and Gradient Boosting on **GazeBase Task identity**. Training uses the participant-disjoint Train records from Notebook 01; reported model scores use Validation records only.

**Statistical unit:** One original recording for model metrics, and one person for partition independence. The four source Task labels are protocol identities, not independent cognitive outcomes. No Validation or Test metadata, source identifiers or recording duration enter the feature matrix.

**Scientific boundary:** Ten-person pilot scores are exploratory. All model settings below are fixed before evaluation. Each model has its own code, record-level metrics and confusion figure; no Test results are inspected here.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_movement_modelling")

# ---------------------------- 2. Data and Models -----------------------------
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from IPython.display import display

# ------------------------- 3. Research Infrastructure ------------------------
from core.stage_outputs import ensure_stage_inputs
from core.output_download import download_outputs_from_colab
from core.plot_style import plot_style, save_figure
from s2_data import TASKS, CLASS_TO_INDEX, load_protocol
from s2_classical import classical_candidates
from s2_metrics import (
    FEATURE_COLUMN_NAMES, require_split_consistency,
    classical_record_predictions, metric_row,
)

# ----------------------------- 4. Output Context -----------------------------
output_dir = root / "eye_movement_modelling" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Current study outputs:", output_dir)


**02. Recover and Check Notebook 01 Feature Inputs**

Read only the source-aligned record feature table, participant split assignments and the study protocol. Restore Notebook 01's explicitly named current ZIP when entering a new Colab runtime.

The source record keys and task identities must match exactly, and each person must belong to a single Train/Validation/Test partition. The Test records are not used here, not even to choose features or imputation values.


In [ ]:
# ======================= 02. Recover Record Features ========================

# ------------------------ 1. Named Current Input Files -----------------------
ensure_stage_inputs(
    output_dir, 1,
    ("01_record_features.csv", "01_split_assignments.csv",
     "01_protocol.json"),
)
features = pd.read_csv(
    output_dir / "01_record_features.csv",
    dtype={"participant_id": str, "record_id": str},
)
split = pd.read_csv(
    output_dir / "01_split_assignments.csv",
    dtype={"participant_id": str, "record_id": str},
)
protocol = load_protocol(output_dir / "01_protocol.json")
require_split_consistency(features, split)

# ---------------------- 2. Separate Train and Validation ---------------------
train = features.loc[features.split.eq("train")].copy()
validation = features.loc[features.split.eq("validation")].copy()
feature_names = list(FEATURE_COLUMN_NAMES)
if train.empty or validation.empty:
    raise ValueError("No Train/Validation task records were selected")
print("Protocol purpose:", protocol["purpose"])
print("Train records / participants:", len(train),
      train.participant_id.nunique())
print("Validation records / participants:", len(validation),
      validation.participant_id.nunique())
print("Input feature columns:", feature_names)
display(train[["record_id", "task", "valid_fraction"]].head(8))
candidates = classical_candidates(seed=protocol["split_seed"])
reported = []


**03. Prior-Only Baseline**

Fit a DummyClassifier that predicts training-set class priors without using gaze features. It provides a minimum reference for the substantive models. Show a separate Validation confusion matrix; a nonzero baseline is not evidence that gaze contains discriminative information.


In [ ]:
# ======================= 03. No-Signal Class-Prior Baseline =======================

# -------------------------- 1. Fit Train Records -----------------------------
name = "class_prior"
model = candidates[name]
model.fit(train[feature_names], train["task"].map(CLASS_TO_INDEX))
joblib.dump(model, output_dir / f"02_{name}.joblib")

# ----------------------- 2. Evaluate Held-Out Validation ---------------------
pred = classical_record_predictions(
    model, validation, model_name=name, partition="validation",
)
pred.to_csv(output_dir / f"02_{name}_validation_predictions.csv", index=False)
row = metric_row(
    pred.target_index, pred.pred_index, model=name,
    partition="validation",
    n_participants=validation["participant_id"].nunique(),
)
reported.append(row)
display(pd.DataFrame([row]))

# ------------------------ 3. Display Model Error Pattern ---------------------
matrix = confusion_matrix(
    pred.target_index, pred.pred_index, labels=range(len(TASKS)),
)
fig, ax = plt.subplots(figsize=(5.6, 4.8))
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=ax, colorbar=False, values_format="d",
)
ax.set_title("Class Prior: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"02_{name}_confusion.png")
plt.show()


**04. Logistic Regression**

Fit a class-balanced, regularised linear model. Median imputation and StandardScaler are estimated **inside the Train-only Pipeline**. Use fixed model settings and evaluate exactly the same Validation records as the prior-only model. Interpretation of coefficients must respect feature collinearity and the small participant count.


In [ ]:
# ======================= 04. Logistic Regression Model =======================

# -------------------------- 1. Fit Train Records -----------------------------
name = "logistic_regression"
model = candidates[name]
model.fit(train[feature_names], train["task"].map(CLASS_TO_INDEX))
joblib.dump(model, output_dir / f"02_{name}.joblib")

# ----------------------- 2. Evaluate Held-Out Validation ---------------------
pred = classical_record_predictions(
    model, validation, model_name=name, partition="validation",
)
pred.to_csv(output_dir / f"02_{name}_validation_predictions.csv", index=False)
row = metric_row(
    pred.target_index, pred.pred_index, model=name,
    partition="validation",
    n_participants=validation["participant_id"].nunique(),
)
reported.append(row)
display(pd.DataFrame([row]))

# ------------------------ 3. Display Model Error Pattern ---------------------
matrix = confusion_matrix(
    pred.target_index, pred.pred_index, labels=range(len(TASKS)),
)
fig, ax = plt.subplots(figsize=(5.6, 4.8))
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=ax, colorbar=False, values_format="d",
)
ax.set_title("Logistic Regression: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"02_{name}_confusion.png")
plt.show()


**05. Random Forest**

Compare a bounded-depth Random Forest with Train-only median imputation and class-balanced fitting. Its feature importance, if inspected later, describes fitted associations rather than causal or physiological explanation. The Validation confusion matrix uses original records as observations.


In [ ]:
# ======================= 05. Random Forest Model =======================

# -------------------------- 1. Fit Train Records -----------------------------
name = "random_forest"
model = candidates[name]
model.fit(train[feature_names], train["task"].map(CLASS_TO_INDEX))
joblib.dump(model, output_dir / f"02_{name}.joblib")

# ----------------------- 2. Evaluate Held-Out Validation ---------------------
pred = classical_record_predictions(
    model, validation, model_name=name, partition="validation",
)
pred.to_csv(output_dir / f"02_{name}_validation_predictions.csv", index=False)
row = metric_row(
    pred.target_index, pred.pred_index, model=name,
    partition="validation",
    n_participants=validation["participant_id"].nunique(),
)
reported.append(row)
display(pd.DataFrame([row]))

# ------------------------ 3. Display Model Error Pattern ---------------------
matrix = confusion_matrix(
    pred.target_index, pred.pred_index, labels=range(len(TASKS)),
)
fig, ax = plt.subplots(figsize=(5.6, 4.8))
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=ax, colorbar=False, values_format="d",
)
ax.set_title("Random Forest: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"02_{name}_confusion.png")
plt.show()


**06. Gradient Boosting**

Fit compact Histogram Gradient Boosting with fixed iterations and tree complexity. The same participant partitions and input feature set are used. Its behaviour can be unstable on 24 Training records in the ten-person pilot; preserve that limitation and do not tune against Test.


In [ ]:
# ======================= 06. Gradient Boosting Model =======================

# -------------------------- 1. Fit Train Records -----------------------------
name = "gradient_boosting"
model = candidates[name]
model.fit(train[feature_names], train["task"].map(CLASS_TO_INDEX))
joblib.dump(model, output_dir / f"02_{name}.joblib")

# ----------------------- 2. Evaluate Held-Out Validation ---------------------
pred = classical_record_predictions(
    model, validation, model_name=name, partition="validation",
)
pred.to_csv(output_dir / f"02_{name}_validation_predictions.csv", index=False)
row = metric_row(
    pred.target_index, pred.pred_index, model=name,
    partition="validation",
    n_participants=validation["participant_id"].nunique(),
)
reported.append(row)
display(pd.DataFrame([row]))

# ------------------------ 3. Display Model Error Pattern ---------------------
matrix = confusion_matrix(
    pred.target_index, pred.pred_index, labels=range(len(TASKS)),
)
fig, ax = plt.subplots(figsize=(5.6, 4.8))
ConfusionMatrixDisplay(matrix, display_labels=TASKS).plot(
    ax=ax, colorbar=False, values_format="d",
)
ax.set_title("Gradient Boosting: Validation by Record")
fig.tight_layout()
save_figure(fig, output_dir / f"02_{name}_confusion.png")
plt.show()


**07. Compare Models on the Same Validation Records**

Report fixed-label Macro F1, Balanced Accuracy and Accuracy, with the number of original Validation records and independent participants. All scores are **record-based**, not window-based. The Validation table supports provisional model selection only; it is not the final evaluation.

This Notebook does not select the final model using Test. Notebook 05 later selects a candidate **only from saved Validation results** and then reports the frozen Test evaluation once.


In [ ]:
# ===================== 07. Compare Classical Validation =====================

# ------------------------ 1. Persist Comparable Scores -----------------------
metrics = pd.DataFrame(reported).sort_values(
    ["macro_f1", "balanced_accuracy", "model"],
    ascending=[False, False, True],
)
metrics.to_csv(output_dir / "02_validation_metrics.csv", index=False)
display(metrics)

# ------------------------- 2. Plot Record-Level Scores -----------------------
fig, ax = plt.subplots(figsize=(8.7, 4.7))
positions = np.arange(len(metrics))
ax.bar(positions - 0.17, metrics["macro_f1"], width=0.34,
       label="Macro F1")
ax.bar(positions + 0.17, metrics["balanced_accuracy"], width=0.34,
       label="Balanced accuracy")
ax.set_xticks(positions, metrics["model"], rotation=20, ha="right")
ax.set(ylabel="Validation score by recording",
       ylim=(0, 1), title="Classical Task Baselines: Validation Only")
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.tight_layout()
save_figure(fig, output_dir / "02_classical_comparison.png")
plt.show()


**08. Interpretation and Local Output Package**

Current outputs include the four fitted joblib Pipeline artefacts, four record-level Validation prediction tables and confusion plots, the common Validation metrics table and a comparative plot. Notebook 05 consumes the saved pipelines as fitted: it does not refit them after inspecting Test.

Notebook 03 and Notebook 04 operate on temporal windows, with the same participant split. This Notebook should not be rerun merely because a later model has a different Validation result.


In [ ]:
# ==================== 08. Export Classical Model Evidence ===================

# --------------------- 1. Confirm Stage-Owned Scientific Files ---------------
print("Comparable validation results:", output_dir / "02_validation_metrics.csv")
for name in candidates:
    print("Fitted pipeline:", output_dir / f"02_{name}.joblib")
    print("Validation confusion:", output_dir / f"02_{name}_confusion.png")

# ---------------------- 2. Package Current Local Outputs ---------------------
download_outputs_from_colab("eye_movement_modelling/output", "02")
